# Research runs: Vaidya et al. (2020) Bi-GRU with attention on 30k comments (Kaggle GPU)

Replication of the model family in Vaidya, Mai and Ning (2020): GloVe embeddings, a bidirectional GRU with attention, optional identity re-weighting and an optional multi-task identity head. Three runs on the same 30k training subset, each followed by per-label threshold tuning and the identity-term bias audit:

| Run | Model | What it tests |
|---|---|---|
| A `vaidya_gru_30k` | single task, weighted BCE | The recurrent baseline |
| B `vaidya_gru_idw3_30k` | A + identity weight 3 | Re-weighting of non-toxic identity mentions |
| C `vaidya_gru_mt_idw3_30k` | B + identity head (multi-task) | Does the auxiliary identity task change the bias metrics? |

**Deviation from the paper:** Jigsaw 2018 has no identity annotations, so the multi-task head predicts weak targets taken from the repo's identity lexicon (does the comment mention a term on each Dixon / Indian identity axis), not human labels.

The first cell of run A downloads GloVe 6B (about 1 GB, cached under `~/.cache/toxicity/embeddings`) and reads the 300d file. Each run should take a few minutes to roughly 15 minutes on a T4.

**Settings:** *Session options* -> Accelerator **GPU T4 x2** or **P100**, Internet **On** (needs a phone-verified Kaggle account).
**Do not** use *Save & Run All* unless you want every cell to run. Download `vaidya_runs.zip` from the *Output* panel at the end.

> Content note: the data contains offensive language.

In [ ]:
REPO_URL = "https://github.com/shriniwas2412/iiith-mml-explainable-toxicity.git"
BRANCH = "main"

import os
if not os.path.exists("repo"):
    !git clone -q -b {BRANCH} {REPO_URL} repo
%cd repo
!git pull -q
!pip install -q -e .
!git rev-parse --short HEAD

In [ ]:
import torch
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn the GPU accelerator on")

## 1. Data, splits and bias templates (about 3 minutes)

In [ ]:
!python scripts/01_download_data.py
!python scripts/03_make_splits.py --subset 30000
!python scripts/07_fetch_bias_resources.py
!python scripts/08_build_templates.py

## 2. Run A: Bi-GRU + attention, weighted BCE

In [ ]:
RUN = "vaidya_gru_30k"
!python scripts/15_vaidya_rnn.py --train data/processed/train_subset_30000.csv --loss weighted_bce --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model rnn:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 3. Run B: + identity re-weighting (weight 3)

In [ ]:
RUN = "vaidya_gru_idw3_30k"
!python scripts/15_vaidya_rnn.py --train data/processed/train_subset_30000.csv --loss weighted_bce --identity-weight 3 --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model rnn:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 4. Run C: + multi-task identity head (weak lexicon targets)

In [ ]:
RUN = "vaidya_gru_mt_idw3_30k"
!python scripts/15_vaidya_rnn.py --train data/processed/train_subset_30000.csv --loss weighted_bce --identity-weight 3 --multitask on --out runs/{RUN}
!python scripts/06_tune_thresholds.py --run runs/{RUN}
!python scripts/09_bias_audit.py --model rnn:runs/{RUN}/model --thresholds runs/{RUN}/thresholds.json --out reports/bias/{RUN}

## 5. Compare all runs and save the results

In [ ]:
!python scripts/12_compare_runs.py
!cat reports/compare/runs_table.md

In [ ]:
# Results without model weights (small). Download it from the Output panel on the right.
!zip -qr /kaggle/working/vaidya_runs.zip runs reports -x "runs/*/model/*" -x "runs/*/model.joblib"
!ls -lh /kaggle/working/*.zip

**After downloading:** unzip `vaidya_runs.zip` inside the repo folder on your laptop, so the files land in `runs/` and `reports/`, then run `python scripts/12_compare_runs.py` there to merge them with the other runs.